# CE 310 — Week 12 In-Class Exercise
## Sensitivity Analysis: Which Input Decides the Answer?

**Dataset:** `CE310_BidItems_2024_2026.csv` — the same file as Week 11
**Points:** 100 (76 auto-graded + 24 manual)

Week 11 fitted a price model and asked whether the differences in it were real.
This week asks a different question: **if the numbers going in are uncertain,
which one of them decides the answer coming out?**

That is the question the slope-stability problem in lecture turned on. Nobody
had measured groundwater, and groundwater was the only input that could move
the factor of safety across the line. The borings everyone wanted to buy would
not have changed the decision.


## Before You Begin

1. Upload `CE310_BidItems_2024_2026.csv` to this Colab session.
2. Run every cell in order. Cells with `___` need you to fill in the blank.
3. Do **not** edit or delete any `print(f"ANSWER_... = ...")` line.


## Setup


In [ ]:
# ── Identify your submission ─────────────────────────────────────────
# Fill both in before you run anything else. NETID is what matches your
# work to your student record — a blank NETID takes a 5-point deduction.
NAME  = ""   # e.g. "Jordan Reyes"
NETID = ""   # e.g. "abc123"

print(f"NAME  = {NAME}")
print(f"NETID = {NETID}")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

items = pd.read_csv('CE310_BidItems_2024_2026.csv')
exc = items[items['ItemDescription'] == 'EXCAV (ROADWAY)'].copy()
exc['LOW']  = (exc['LowBidder'] == 'Yes').astype(int)
exc['logq'] = np.log10(exc['Quantity'])
exc['logp'] = np.log10(exc['UnitPrice_USD'])

# Model 2 — the Week 11 price model, with the low-bidder indicator added
m2 = smf.ols('logp ~ logq + LOW', data=exc).fit()
print(m2.summary().tables[1])


## Section A — Analytical and Numerical Sensitivity

Section A works one line item, roadway excavation, at a **design quantity of
2,000 cubic yards** — a routine job, close to the median of 2,137 CY.


### A1 — Baseline Prediction at the Design Point

Everything downstream is a change *from* this number, so compute it first.
The model predicts log₁₀ price, so raise 10 to that power to get dollars.


In [ ]:
Q0 = 2000          # design quantity, cubic yards
LOW0 = 1           # price a winning bid

logP = (m2.params['Intercept']
        + m2.params['logq'] * np.log10(___)   # fill in: the design quantity
        + m2.params['LOW']  * LOW0)
P0 = ___ ** logP                              # fill in: undo the log10

print(f'design quantity   {Q0:,} CY')
print(f'predicted price   ${P0:.4f} per CY')
print()

ANSWER_A1_base_price = round(P0, 2)
print(f'ANSWER_A1_base_price = {ANSWER_A1_base_price}')


### A2 — The Slope IS the Sensitivity

For a fitted model you do not need to re-run anything to know how the output
responds — the coefficient already tells you.

On a log–log scale the `logq` coefficient is an **elasticity**: the percentage
change in price per one percent change in quantity. Report it, then convert it
to dollars per CY at the design point using

$$\frac{dP}{dQ} = \beta_{logq} \cdot \ln(10) \cdot \frac{P}{Q}$$


In [ ]:
elasticity = m2.params['logq']
dP_dQ = elasticity * np.log(10) * ___ / ___   # fill in: from the formula above

print(f'elasticity (logq coefficient) = {elasticity:.4f}')
print(f'  -> a 10x larger job prices {abs(100*(10**elasticity - 1)):.1f}% lower per CY')
print(f'dP/dQ at the design point     = ${dP_dQ:.6f} per CY, per additional CY')
print()

ANSWER_A2_elasticity = round(elasticity, 4)
print(f'ANSWER_A2_elasticity = {ANSWER_A2_elasticity}')


### A3 — Numerical OAT Check

Analytical sensitivity is exact only for an infinitesimal change. Engineers
work with finite ones. Move quantity **+15%** — a routine take-off error — and
measure what actually happens.


In [ ]:
def price(q, low=1, params=None):
    p = m2.params if params is None else params
    return 10 ** (p['Intercept'] + p['logq'] * np.log10(q) + p['LOW'] * low)

P_up = price(___)          # fill in: the design quantity raised by 15%
pct  = 100 * (P_up / P0 - 1)

print(f'price at {Q0:,} CY = ${P0:.4f}')
print(f'price at +15%     = ${P_up:.4f}')
print(f'change = {pct:.4f}%')
print()

ANSWER_A3_pct_change = round(pct, 2)
print(f'ANSWER_A3_pct_change = {ANSWER_A3_pct_change}')


> A 15% rise in quantity moves the unit price only about 2.3% — because the
> elasticity is small. Note the sign: **more work, lower unit price.**


### A4 — Programme Aggregate

One job's sensitivity is not the number an agency budgets against. Scale to the
whole awarded programme.

**Careful with the aggregate.** Each project appears once per bidder in this
file, so summing `Quantity` over all rows counts every project about five
times. Aggregate over **awarded work only** — the winning bids.


In [ ]:
won = exc[exc['LowBidder'] == ___]   # fill in: keep awarded work only

prog_qty  = won[___].sum()           # fill in: the column being aggregated
prog_cost = (won['Quantity'] * won['UnitPrice_USD']).sum()

print(f'awarded projects     {won["ProjectID"].nunique():,}')
print(f'programme quantity   {prog_qty:,.0f} CY')
print(f'programme cost       ${prog_cost:,.0f}')
print()

ANSWER_A4_prog_qty = int(prog_qty)
print(f'ANSWER_A4_prog_qty = {ANSWER_A4_prog_qty}')


### A5–A7 — Does the Sensitivity Depend on Who Is Bidding?

Model 2 gives winners and losers the same slope and shifts one against the
other. The **interaction model** lets the slope itself differ.


In [ ]:
m5 = smf.ols(___, data=exc).fit()   # fill in: the interaction formula, * not +
print(m5.summary().tables[1])

def price5(q, low):
    p = m5.params
    return 10 ** (p['Intercept'] + p['logq'] * np.log10(q)
                  + p['LOW'] * low + p['logq:LOW'] * np.log10(q) * low)

P_win  = price5(Q0, ___)   # fill in: the LOW indicator for a winning bid
P_lose = price5(Q0, ___)   # fill in: and for a losing one

print()
print(f'winning bid at {Q0:,} CY = ${P_win:.4f}')
print(f'losing  bid at {Q0:,} CY = ${P_lose:.4f}')
print(f'ratio = {P_lose / P_win:.4f}')
print()

ANSWER_A5_price_win  = round(P_win, 2)
ANSWER_A7_ratio      = round(P_lose / P_win, 3)
print(f'ANSWER_A5_price_win = {ANSWER_A5_price_win}')
print(f'ANSWER_A7_ratio = {ANSWER_A7_ratio}')


### A8 — The Swing a Take-off Error Produces

A **swing** is the distance between the low and high ends of an input's
plausible range. Compute the price swing for a take-off error of ±15%.


In [ ]:
Q_hi = ___   # fill in: the design quantity at +15%
Q_lo = ___   # fill in: and at -15%
swing_qty = abs(price(Q_hi) - price(Q_lo))

print(f'price at -15% = ${price(Q_lo):.4f}')
print(f'price at +15% = ${price(Q_hi):.4f}')
print(f'swing         = ${swing_qty:.4f} per CY')
print()

ANSWER_A8_swing_qty = round(swing_qty, 3)
print(f'ANSWER_A8_swing_qty = {ANSWER_A8_swing_qty}')


## Section B — Ranking the Inputs

### B1 — Interpretation (Manual · 4 pts)

In 3–4 sentences, explain what A2 and A3 together tell you about this model's
sensitivity to quantity, using both numbers. Then state why an elasticity of
−0.166 does **not** mean quantity is unimportant to the cost of a job.


*Your answer:*


### B2 — Tornado Diagram: Four Inputs, One Cost (chart · Manual · 4 pts)

A bid item's cost is not just quantity times price. Two more inputs enter after
the price model has done its work:

$$\text{item cost} = Q \times P \times (1 + \text{overhead}) \times (1 + \text{escalation})$$

Each carries a plausible range, and the ranges are not equally wide:

| Input | Low | Base | High | Where the range comes from |
|---|---|---|---|---|
| Quantity | −15% | 2,000 CY | +15% | a routine take-off dispute |
| Unit price | −1 RMSE | model | +1 RMSE | the fitted model's own residual spread |
| Overhead | 8% | 13% | 18% | variation across firms |
| Escalation | 2% | 5% | 8% | depends on contract duration |


In [ ]:
RMSE = np.sqrt(___)   # fill in: the m2 attribute holding the residual variance
print(f'model RMSE = {RMSE:.4f} log10 units -> price band x{10**RMSE:.2f} / x{10**-RMSE:.2f}')

def cost(q, p, oh, esc):
    return q * p * (1 + oh) * (1 + esc)

BASE = dict(q=Q0, p=P0, oh=0.13, esc=0.05)
base_cost = cost(**BASE)

RANGES = {
    'Unit price':  ('p',   P0 * 10**-RMSE, P0 * 10**RMSE),
    'Quantity':    ('q',   Q0 * ___,       Q0 * ___),      # fill in: -15%, +15%
    'Overhead':    ('oh',  0.08,           0.18),
    'Escalation':  ('esc', 0.02,           0.08),
}

swings = {}
for label, (key, lo, hi) in RANGES.items():
    a = dict(BASE); a[key] = lo
    b = dict(BASE); b[key] = hi
    swings[label] = abs(cost(**b) - cost(**a))

print(f'\nbase cost = ${base_cost:,.0f}\n')
for label, s in sorted(swings.items(), key=lambda x: -x[1]):
    print(f'  {label:<12} swing ${s:>10,.0f}   ({100*s/base_cost:5.1f}% of base)')
print()

ANSWER_B2_swing_price = round(swings['Unit price'], 0)
ANSWER_B2_swing_qty   = round(swings['Quantity'], 0)
ANSWER_B2_swing_oh    = round(swings['Overhead'], 0)
print(f'ANSWER_B2_swing_price = {ANSWER_B2_swing_price}')
print(f'ANSWER_B2_swing_qty = {ANSWER_B2_swing_qty}')
print(f'ANSWER_B2_swing_oh = {ANSWER_B2_swing_oh}')


In [ ]:
order = sorted(swings, key=swings.get)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(order, [swings[k] for k in order], color='#0C234B', alpha=0.85)
ax.set_xlabel('swing in item cost, dollars')
ax.set_title(f'Tornado — {Q0:,} CY excavation, base ${base_cost:,.0f}')
for i, k in enumerate(order):
    ax.text(swings[k], i, f'  ${swings[k]:,.0f}', va='center', fontsize=10)
ax.margins(x=0.18)
plt.tight_layout()
plt.show()


> **Read the ranking before you go on.** Cost is *linear* in all four inputs —
> none of them has a steeper response than any other. Unit price wins purely
> because it is the input we know least well. That is the lecture's slope
> problem again: rank by response **times** uncertainty, never by response alone.


### B3 — Would a Better Estimate Change the Ranking?

Suppose a detailed quantity survey narrows the take-off range from ±15% to ±3%.
Recompute that swing and say whether the ranking changes.


In [ ]:
tight = abs(cost(Q0*___, P0, 0.13, 0.05) - cost(Q0*___, P0, 0.13, 0.05))   # +/-3%

print(f'quantity swing at +/-15% = ${swings["Quantity"]:,.0f}')
print(f'quantity swing at +/-3%  = ${tight:,.0f}')
print(f'overhead swing (unchanged) = ${swings["Overhead"]:,.0f}')
print()

ANSWER_B3_tight_swing = round(tight, 0)
print(f'ANSWER_B3_tight_swing = {ANSWER_B3_tight_swing}')


### B4 — Programme-Level Exposure

Scale the dominant input to the whole awarded programme. If every unit price
were off by one RMSE in the same direction, what would that do to the
programme cost from A4?


In [ ]:
prog_exposure = prog_cost * (10**___ - 1)   # fill in: one RMSE, in log10 units

print(f'programme cost      ${prog_cost:,.0f}')
print(f'+1 RMSE on price    ${prog_exposure:,.0f}  ({100*prog_exposure/prog_cost:.1f}%)')
print()

ANSWER_B4_prog_exposure = round(prog_exposure, 0)
print(f'ANSWER_B4_prog_exposure = {ANSWER_B4_prog_exposure}')


### B5 — Overhead Sensitivity per Point

Agencies negotiate overhead. Compute the cost change per **one percentage
point** of overhead, at the design job.


In [ ]:
per_point = cost(Q0, P0, ___, 0.05) - cost(Q0, P0, ___, 0.05)   # fill in: one point above base, and base

print(f'cost per +1 point of overhead = ${per_point:,.2f}')
print()

ANSWER_B5_per_point = round(per_point, 2)
print(f'ANSWER_B5_per_point = {ANSWER_B5_per_point}')


### B6 — Design Uncertainty as a Percentage

Express the total spread — cheapest plausible corner against dearest — as a
percentage of the base cost. This is the number a contingency decision is made
against.


In [ ]:
lo_corner = cost(Q0*0.85, P0*10**-RMSE, ___, ___)   # fill in: low overhead, low escalation
hi_corner = cost(Q0*1.15, P0*10**RMSE,  ___, ___)   # fill in: and the high end of each
spread_pct = 100 * (hi_corner - lo_corner) / base_cost

print(f'all-low corner   ${lo_corner:,.0f}')
print(f'base             ${base_cost:,.0f}')
print(f'all-high corner  ${hi_corner:,.0f}')
print(f'spread = {spread_pct:.1f}% of base')
print()

ANSWER_B6_spread_pct = round(spread_pct, 1)
print(f'ANSWER_B6_spread_pct = {ANSWER_B6_spread_pct}')


> **Note what you just did.** Moving every input to its worst corner at once is
> not a 95% case — it assumes all four go wrong together. Week 13 replaces this
> corner-stacking with sampling, and the difference between the two is the
> reason Monte Carlo exists.


### B7 — A Larger Job

Repeat the tornado at **50,000 CY**, a major earthwork job. Report the new
unit price and say whether the ranking of the four inputs changed.

**Written response (Manual · 4 pts).** Below the code cell: give the new unit
price against the $21.26 at 2,000 CY, state whether the ranking changed, support
it with the swings as **percentages of base** rather than dollars, and say what
scaling the job does to every swing at once.


In [ ]:
Q1 = 50000
P1 = price(___)   # fill in: the unit price at the larger job
base1 = cost(Q1, P1, 0.13, 0.05)

sw1 = {}
for label, (key, lo, hi) in RANGES.items():
    b = dict(q=Q1, p=P1, oh=0.13, esc=0.05)
    if key == 'q':   lo, hi = Q1*0.85, Q1*1.15
    if key == 'p':   lo, hi = P1*10**-RMSE, P1*10**RMSE
    a = dict(b); a[key] = lo
    c = dict(b); c[key] = hi
    sw1[label] = abs(cost(**c) - cost(**a))

print(f'unit price at {Q1:,} CY = ${P1:.4f}   base cost ${base1:,.0f}\n')
for label, s in sorted(sw1.items(), key=lambda x: -x[1]):
    print(f'  {label:<12} ${s:>12,.0f}   ({100*s/base1:5.1f}% of base)')
print()

ANSWER_B7_price_50k = round(P1, 2)
print(f'ANSWER_B7_price_50k = {ANSWER_B7_price_50k}')


*Your answer:*

### B8 — How Accurate Does the Take-off Need to Be?

Turn the tornado into a specification. Find the quantity tolerance, in cubic
yards, at which the take-off swing drops below the overhead swing — the point
past which more survey effort stops changing the ranking.


In [ ]:
tol_CY = swings[___] / (2 * P0 * ___)   # fill in: the swing to match, and the two markups

print(f'overhead swing        ${swings["Overhead"]:,.0f}')
print(f'quantity tolerance    +/-{tol_CY:.1f} CY  ({100*tol_CY/Q0:.2f}% of {Q0:,} CY)')
print()

ANSWER_B8_tol_CY = round(tol_CY, 1)
print(f'ANSWER_B8_tol_CY = {ANSWER_B8_tol_CY}')


### B9 — Written Reflection (Manual · 4 pts)

In 100–150 words: the lecture's slope problem ranked groundwater above friction
angle even though both affect the factor of safety, because nobody had measured
groundwater. State which input in **your** tornado plays that role and why.
Then name one thing a project team could do that would change the ranking, and
one that would not — with a number from B3 or B8 supporting each.


*Your answer:*


## Memo (Manual · 8 pts)

Write a short technical memo (To / From / Date / Subject, then three paragraphs)
to an estimating manager who has asked where to spend a limited budget on
improving cost forecasts.

- **Paragraph 1** — what the tornado shows. Cite the base cost and the four
  swings from B2, and say plainly which input dominates.
- **Paragraph 2** — what would and would not help. Use B3 (a tighter take-off)
  and B8 (the tolerance at which more survey stops mattering).
- **Paragraph 3** — one recommendation, and one honest limitation. The corner
  stacking in B6 is the obvious candidate: it is not a 95% case, and saying so
  is more useful than quoting it as one.


## Looking Ahead: Week 13 — Monte Carlo Simulation

B6 moved every input to its worst corner at once and produced a number no
sensible estimator would budget against, because all four going wrong together
is far less likely than any one of them doing so. Week 13 samples the same four
inputs from distributions instead of sweeping them, and reports what fraction of
simulated jobs land where. Same model, same inputs, an answer you can defend.


## Before You Submit

- [ ] `NAME` and `NETID` filled in at the top and showing in the cell output — a blank `NETID` takes a 5-point deduction
- [ ] Every cell run in order, top to bottom (Runtime → Run all) — no errors, no cell left unrun
- [ ] Every `ANSWER_` line prints a value, and no `print(f"ANSWER_... = ...")` line edited or deleted
- [ ] Every `___` replaced
- [ ] The B2 tornado chart plotted, sorted, all four bars labelled
- [ ] B1, B7, B9 and the memo written in their markdown cells
- [ ] Downloaded as `.ipynb` — not PDF, not `.py`
- [ ] Renamed `CE310_W12_<NetID>.ipynb` and uploaded to the Week 12 D2L dropbox